# Lab 1: Vacuum Cleaner Agent (4 rooms)
Environment (Anaconda): `conda create -n ailab python=3.11 numpy matplotlib jupyter`, then `conda activate ailab` and `jupyter notebook`.

Run cells top to bottom with **Shift+Enter**.

**PEAS:** performance = +1 per clean room per step; environment = 4 rooms with random dirt; actuators = Left/Right/Suck/NoOp; sensors = (position, status).

## Cell 1: Setup (ready-made libraries)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

N_ROOMS = 4     # number of rooms
STEPS = 50      # steps in one run
RUNS = 100      # number of experiments per agent

## Cell 2: One simple function

In [ ]:
def simulate(agent, seed):
    rng = np.random.default_rng(seed)
    dirt = rng.random(N_ROOMS) < 0.5               # True = dirty room
    known_clean = np.zeros(N_ROOMS, dtype=bool)    # memory of the "model" agent
    pos, direction, score = 0, 1, 0

    for step in range(STEPS):
        known_clean[pos] = not dirt[pos]

        # 1) choose an action
        if agent == "random":
            action = rng.choice(["Left", "Right", "Suck"])
        elif dirt[pos]:
            action = "Suck"
        elif agent == "reflex":
            action = rng.choice(["Left", "Right"])
        elif known_clean.all():
            action = "NoOp"
        else:
            if pos == N_ROOMS - 1:
                direction = -1
            if pos == 0:
                direction = 1
            action = "Right" if direction == 1 else "Left"

        # 2) apply the action
        if action == "Suck":
            dirt[pos] = False
        elif action == "Right":
            pos = min(pos + 1, N_ROOMS - 1)
        elif action == "Left":
            pos = max(pos - 1, 0)

        score += (~dirt).sum()                     # +1 for each clean room
    return int(score)

## Cell 3: Quick test of the 3 agents

In [ ]:
for agent in ["random", "reflex", "model"]:
    print(agent, simulate(agent, seed=1))

## Cell 4: 100 experiments (pandas)

In [ ]:
rows = []
for agent in ["random", "reflex", "model"]:
    for seed in range(RUNS):
        rows.append({"agent": agent, "score": simulate(agent, seed)})

df = pd.DataFrame(rows)
print(df.head())

## Cell 5: Summary table

In [ ]:
summary = df.groupby("agent")["score"].agg(["mean", "std"]).round(1)
print(summary)

## Cell 6: Plot

In [ ]:
summary["mean"].plot(kind="bar", yerr=summary["std"], capsize=5)
plt.title("Average score over 100 runs (4 rooms, 50 steps)")
plt.ylabel("score")
plt.xticks(rotation=0)
plt.show()

## Analysis questions
1. Why is the random agent weak?
2. Why does the model-based agent stop?
3. What if dirt reappears with probability p?

## Challenges
2D grid, noisy sensor, energy cost, dirt reappearing.

## Cell 7: Record frames for animation

This function follows the same agent rules as `simulate`, but saves the room state after every step so it can be animated.

In [ ]:
def record(agent, seed=1):
    rng = np.random.default_rng(seed)
    dirt = rng.random(N_ROOMS) < 0.5
    known_clean = np.zeros(N_ROOMS, dtype=bool)
    pos, direction, score = 0, 1, 0
    frames = [(pos, dirt.copy(), "Start", score)]

    for step in range(STEPS):
        known_clean[pos] = not dirt[pos]

        if agent == "random":
            action = rng.choice(["Left", "Right", "Suck"])
        elif dirt[pos]:
            action = "Suck"
        elif agent == "reflex":
            action = rng.choice(["Left", "Right"])
        elif known_clean.all():
            action = "NoOp"
        else:
            if pos == N_ROOMS - 1:
                direction = -1
            if pos == 0:
                direction = 1
            action = "Right" if direction == 1 else "Left"

        if action == "Suck":
            dirt[pos] = False
        elif action == "Right":
            pos = min(pos + 1, N_ROOMS - 1)
        elif action == "Left":
            pos = max(pos - 1, 0)

        score += int((~dirt).sum())
        frames.append((pos, dirt.copy(), action, score))

    return frames


## Cell 8: Animate the three agents

Each row shows one agent moving through the same initial room layout. Red rooms are dirty; green rooms are clean.

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

names = ["random", "reflex", "model"]
runs = [record(agent, seed=1) for agent in names]

fig, axes = plt.subplots(3, 1, figsize=(7, 4.5), dpi=80)

def draw(t):
    for ax, name, frames in zip(axes, names, runs):
        pos, dirt_state, action, score = frames[t]
        ax.clear()
        ax.set_xlim(0, N_ROOMS)
        ax.set_ylim(0, 1)
        ax.axis("off")

        for room in range(N_ROOMS):
            color = "#fecaca" if dirt_state[room] else "#bbf7d0"
            ax.add_patch(plt.Rectangle((room + 0.05, 0.05), 0.9, 0.9, color=color))
            if dirt_state[room]:
                ax.add_patch(plt.Circle((room + 0.5, 0.35), 0.13, color="#38bdf8"))

        ax.text(pos + 0.5, 0.68, "R", fontsize=19, fontweight="bold",
                ha="center", va="center")
        ax.set_title(f"{name} | step {t} | action: {action} | score: {score}", fontsize=9)

anim = FuncAnimation(fig, draw, frames=len(runs[0]), interval=250, repeat=True)
plt.close(fig)
HTML(anim.to_jshtml())


## Cell 9: Animated bar chart of average scores

This animation grows the bars toward the final mean scores computed in Cell 5.

In [ ]:
fig2, ax2 = plt.subplots(figsize=(6, 3.5), dpi=80)
final = summary["mean"]

def grow(k):
    ax2.clear()
    ax2.set_ylim(0, max(210, float(final.max()) * 1.15))
    ax2.set_title("Average score over 100 runs")
    ax2.set_ylabel("Score")
    values = final.values * (k + 1) / 20
    bars = ax2.bar(final.index, values, color=["#ec4899", "#4f46e5", "#f59e0b"])
    for bar, value in zip(bars, values):
        ax2.text(bar.get_x() + bar.get_width() / 2, value + 2,
                 f"{value:.1f}", ha="center", va="bottom")
    ax2.set_xticks(range(len(final.index)), final.index, rotation=0)

anim2 = FuncAnimation(fig2, grow, frames=20, interval=120, repeat=False)
plt.close(fig2)
HTML(anim2.to_jshtml())


---
# Development challenge: Dirt reappears → Preferences change

**This section is added to the instructor's original notebook.** The original 4-room simulation, charts, and animations remain unchanged above.

In a dynamic environment, dirt can reappear even after rooms have been cleaned. The original `model` agent stops moving after its internal memory says all rooms are clean. We compare this behavior with `model_adaptive`, which continues visiting rooms to detect new dirt.

**Link to the final project:** A movie/book recommender should likewise reconsider its decisions when user tastes or available items change.

In [ ]:
def simulate_dynamic(agent, seed, redirty_prob=0.06, steps=STEPS, return_history=False):
    """Extend the original classroom simulation with reappearing dirt.

    The environmental randomness is pre-generated independently of agent actions.
    This makes comparisons across agents fair for a fixed seed.
    """
    if not 0 <= redirty_prob <= 1:
        raise ValueError("redirty_prob must be between 0 and 1")
    if agent not in {"random", "reflex", "model", "model_adaptive"}:
        raise ValueError("Unknown agent")

    env_rng = np.random.default_rng(seed)
    action_rng = np.random.default_rng(seed + 100_000)
    dirt = env_rng.random(N_ROOMS) < 0.5
    redirty_rolls = env_rng.random((steps, N_ROOMS))
    known_clean = np.zeros(N_ROOMS, dtype=bool)
    pos, direction, score = 0, 1, 0
    history = []

    for t in range(steps):
        known_clean[pos] = not dirt[pos]   # only the current room can be observed

        if agent == "random":
            action = str(action_rng.choice(["Left", "Right", "Suck"]))
        elif dirt[pos]:
            action = "Suck"
        elif agent == "reflex":
            action = str(action_rng.choice(["Left", "Right"]))
        elif agent == "model" and known_clean.all():
            action = "NoOp"               # existing model's stopping decision
        else:
            # model_adaptive keeps patrolling; base model scans until all known clean
            if pos == N_ROOMS - 1:
                direction = -1
            elif pos == 0:
                direction = 1
            action = "Right" if direction == 1 else "Left"

        if action == "Suck":
            dirt[pos] = False
            known_clean[pos] = True
        elif action == "Right":
            pos = min(pos + 1, N_ROOMS - 1)
        elif action == "Left":
            pos = max(pos - 1, 0)

        # A clean room can become dirty again after the action.
        reappeared = (~dirt) & (redirty_rolls[t] < redirty_prob)
        dirt[reappeared] = True
        score += int((~dirt).sum())
        if return_history:
            history.append({"step": t + 1, "action": action, "position": pos,
                            "clean_rooms": int((~dirt).sum()),
                            "new_dirt_count": int(reappeared.sum()), "score": score})

    return (int(score), pd.DataFrame(history)) if return_history else int(score)

## Compare agents under a static and changing environment

We keep the original conditions (4 rooms, 50 steps, 100 seeds), adding a fixed re-dirty probability. The maximum score per run is 4 × 50 = 200. A higher average score means more clean rooms over time.

In [ ]:
comparison_rows = []
for prob in [0.00, 0.06]:
    for agent in ["random", "reflex", "model", "model_adaptive"]:
        for seed in range(RUNS):
            comparison_rows.append({
                "redirty_probability": prob,
                "agent": agent,
                "score": simulate_dynamic(agent, seed, redirty_prob=prob),
            })
comparison = pd.DataFrame(comparison_rows)
comparison_summary = (
    comparison.groupby(["redirty_probability", "agent"])["score"]
    .agg(["mean", "std"])
    .round(2)
)
print(comparison_summary)

ax = comparison_summary["mean"].unstack(level=0).plot.bar(
    figsize=(8, 4), rot=0, title="Performance with and without reappearing dirt")
ax.set_ylabel("Mean clean-room score (max 200)")
ax.set_xlabel("Agent")
plt.tight_layout()
plt.show()

In [ ]:
# Inspect a few time steps for a single run; compare NoOp with periodic patrol.
for name in ["model", "model_adaptive"]:
    score, trace = simulate_dynamic(name, seed=3, redirty_prob=0.06, return_history=True)
    print(f"\n{name}: score={score}, number of NoOp steps={(trace['action'] == 'NoOp').sum()}")
    print(trace.tail(5).to_string(index=False))

# Miniature movie recommender (connection to the final project)

This self-contained demonstration is **not** a trained recommender. We use a tiny fictional movie catalog and simple genre weights to show what the same dynamic-environment idea means for movie or book recommendations.

- Static strategy: the initial taste profile stays unchanged.
- Adaptive strategy: explicit new feedback updates genre scores; new content can also enter the catalog.
- Illustrative metric: Precision@3, the fraction of the three suggestions matching a scripted currently preferred genre. This is a simulation metric, not a real-world accuracy claim.

In [ ]:
movie_catalog = [
    {"title": "Movie 01", "genre": "Action"},
    {"title": "Movie 02", "genre": "Action"},
    {"title": "Movie 03", "genre": "Action"},
    {"title": "Movie 04", "genre": "Sci-Fi"},
    {"title": "Movie 05", "genre": "Sci-Fi"},
    {"title": "Movie 06", "genre": "Comedy"},
    {"title": "Movie 07", "genre": "Comedy"},
    {"title": "Movie 08", "genre": "Drama"},
]
initial_profile = {"Action": 3, "Sci-Fi": 2, "Comedy": 0, "Drama": 0}
static_profile = initial_profile.copy()
adaptive_profile = initial_profile.copy()

def recommend(catalog, profile, k=3):
    return sorted(catalog, key=lambda item: (-profile.get(item["genre"], 0), item["title"]))[:k]

def precision_at_k(recommendations, preferred_genres):
    if not recommendations:
        return 0.0
    return sum(item["genre"] in preferred_genres for item in recommendations) / len(recommendations)

def show_recommendations(label, catalog, profile, preferred_genres):
    items = recommend(catalog, profile)
    print(f"{label}: {[item['title'] + ' (' + item['genre'] + ')' for item in items]}")
    print(f"  Precision@{len(items)} (illustrative): {precision_at_k(items, preferred_genres):.2f}")

print("Before the preference change:")
show_recommendations("Static", movie_catalog, static_profile, {"Action", "Sci-Fi"})
show_recommendations("Adaptive", movie_catalog, adaptive_profile, {"Action", "Sci-Fi"})

In [ ]:
# User taste changes toward Comedy, and two new comedy movies become available.
new_feedback = [("Comedy", +4), ("Action", -4)]
for genre, value in new_feedback:
    adaptive_profile[genre] = adaptive_profile.get(genre, 0) + value

movie_catalog = movie_catalog + [
    {"title": "Movie 09 (new)", "genre": "Comedy"},
    {"title": "Movie 10 (new)", "genre": "Comedy"},
]

print("After the preference change and new arrivals:")
show_recommendations("Static", movie_catalog, static_profile, {"Comedy"})
show_recommendations("Adaptive", movie_catalog, adaptive_profile, {"Comedy"})
print("Adaptive genre scores:", adaptive_profile)

## Interpretation and limitations

The **reappearing dirt** and **changing tastes** examples both require reevaluating a state that was assumed to be settled. In the first, the model-based vacuum agent must revisit cleaned rooms. In the second, the recommender must update its preference profile when the user provides feedback and consider new items.

**Limits:** The catalog and feedback are fictional, the genre-based recommender is rule-based rather than machine-learned, and Precision@3 is measured against a scripted preference. Future steps should use real user interactions/datasets, prevent recommending already-consumed items, and compare to baselines. The vacuum experiment's score and the recommendation metric represent different tasks and must not be treated as interchangeable.